In [2]:
import os
import re
import pandas as pd
from datetime import datetime

# 1. Konfigurasi Folder
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Stopwords dasar Bahasa Indonesia untuk membantu pembersihan teks feature
INDONESIAN_STOPWORDS = set(['yang', 'dan', 'di', 'dari', 'untuk', 'pada', 'ke', 'oleh', 'sebagaimana', 'bahwa', 'tersebut'])

def advanced_cleaner(text):
    """Fungsi pembersihan teks tingkat lanjut khusus dokumen hukum MA"""
    # Catat teks awal untuk validasi keutuhan
    if len(text.strip()) == 0:
        return ""
    
    # Lowercase & Hapus watermark/header khas Direktori Putusan MA
    text = text.lower()
    text = re.sub(r'direktori putusan mahkamah agung republik indonesia', '', text)
    text = re.sub(r'halaman \d+ dari \d+ halaman', '', text)
    text = re.sub(r'catatan catatan', '', text)
    
    # Hapus karakter khusus & angka penting yang tidak merepresentasikan kata (opsional)
    text = re.sub(r'[^\w\s\-\/\.]', '', text)
    
    # Normalisasi spasi ganda menjadi satu spasi
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_metadata(text):
    """Mengekstrak informasi penting secara otomatis menggunakan RegEx"""
    # Cari Nomor Perkara
    no_perkara_pattern = r'nomor\s+(\d+/[a-z]+\.[a-z]+/\d+/pn\s+[a-z]+)'
    no_perkara_match = re.search(no_perkara_pattern, text)
    no_perkara = no_perkara_match.group(1).upper() if no_perkara_match else "N/A"
    
    # Cari Pasal (Contoh makro untuk Kasus Narkotika/Wanprestasi)
    pasal_pattern = r'(pasal\s+\d+\s+uu\s+nomor\s+\d+\s+tahun\s+\d+|pasal\s+\d+\s+kuhper|pasal\s+\d+\s+kuhp)'
    pasal_match = re.search(pasal_pattern, text)
    pasal = pasal_match.group(1) if pasal_match else "Pasal tidak terdeteksi"
    
    return no_perkara, pasal

# ==========================================
# SIMULASI EKSEKUSI PIPELINE (30+ DOKUMEN)
# ==========================================
# Catatan: Ganti 'dataset_mentah' dengan dictionary hasil scraping BeautifulSoup Anda
dataset_mentah = {
    f"case_{str(i).zfill(3)}": f"Putusan NOMOR {i}/Pid.Sus/2023/PN Mlg. Mengadili bahwa Terdakwa terbukti melanggar Pasal 112 UU Nomor 35 Tahun 2009 tentang Narkotika. Membawa sabu seberat {i*0.5} gram. Menjatuhkan hukuman penjara." 
    for i in range(1, 35) # Membuat 34 dokumen dummy untuk simulasi aman > 30 dokumen
}

cases_data = []
log_entries = []

for case_id, raw_text in dataset_mentah.items():
    # Validasi keutuhan (Minimal 80% isi tersedia / tidak kosong)
    if len(raw_text) < 50:
        log_entries.append(f"{datetime.now()} - [SKIP] {case_id} gagal divalidasi karena teks terlalu pendek.\n")
        continue
        
    # Tahap 1: Pembersihan Teks
    cleaned_text = advanced_cleaner(raw_text)
    
    # Simpan teks bersih secara modular ke /data/raw/
    with open(os.path.join(RAW_DIR, f"{case_id}.txt"), 'w', encoding='utf-8') as f:
        f.write(cleaned_text)
        
    # Tahap 2: Ekstraksi Fitur & Metadata
    no_perkara, pasal = extract_metadata(cleaned_text)
    
    # Pemisahan Konten Kunci (Ringkasan Fakta vs Solusi)
    words = cleaned_text.split()
    ringkasan_fakta = " ".join(words[:len(words)//2]) # Setengah teks awal sebagai fakta
    amar_putusan = " ".join(words[len(words)//2:])    # Setengah teks akhir sebagai amar/solusi
    
    cases_data.append({
        'case_id': case_id,
        'no_perkara': no_perkara,
        'tanggal': '2023-05-12', # Bisa dinormalisasi lebih lanjut
        'pihak': 'Jaksa Penuntut Umum vs Terdakwa',
        'pasal': pasal,
        'ringkasan_fakta': ringkasan_fakta,
        'amar_putusan_solusi': amar_putusan,
        'text_full': cleaned_text
    })
    log_entries.append(f"{datetime.now()} - [SUCCESS] {case_id} berhasil diproses dan disimpan.\n")

# Tulis file log otomatis
os.makedirs('../logs/', exist_ok=True)
with open('../logs/cleaning.log', 'w') as log_file:
    log_file.writelines(log_entries)

# Simpan hasil akhir ke CSV terstruktur
df_cases = pd.DataFrame(cases_data)
df_cases.to_csv(os.path.join(PROCESSED_DIR, 'cases.csv'), index=False)
print(f"Pipeline Sukses! 30+ Data disimpan di '{PROCESSED_DIR}cases.csv' dan Log diterbitkan.")

Pipeline Sukses! 30+ Data disimpan di '../data/processed/cases.csv' dan Log diterbitkan.
